# debug_rmmv5 — GDN / memory head & state-size safety net

**Do not trust prose claims** about which knobs reach the model. Rebuild the exact
model the run scripts build, then read the *actual* heads/dims/state off the live modules.

- **v6p2 (buggy):** GDN ignores `state_size`/`num_heads`/`head_dim` → FLA defaults; ss16 ≡ ss32.
- **v5p7 (control):** uses the real `GatedDeltaNet` class → `head_dim = state_size//n_head` IS honored.
- **v6p3 (fixed):** knobs honored + hard-errors on silent drops + logs resolved GDN geometry.

In [16]:
import os, sys, inspect
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())=="notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("cwd:", os.getcwd())
import torch
from transformers import AutoConfig, AutoTokenizer
TOKENIZER_PATH = "./tokenizers/kv_alphabet_62/"   # as in run_rmm_on_kv_retrieval-v6p2.py

cwd: /home/bulatov/rmt/test-time/compressing-associations-gdn


In [17]:
def build_model(state_size=32, n_head=4, n_embd=128, n_layer=4,
                num_memory_vectors=32, write_mode="pool", read_mode="identity",
                num_memory_heads=1, expand_v=2.0, conv_kernel=4,
                fla_layer_name="GatedDeltaNet", module="v6p2"):
    """Replicates run_rmm_on_kv_retrieval-v6p2.py model creation (llama base).
    `module` selects the modeling file (v5p7 / v6p2 / v6p3)."""
    tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    config = AutoConfig.from_pretrained("NousResearch/Llama-3.2-1B")
    config.num_hidden_layers   = n_layer
    config.num_attention_heads = n_head
    config.num_key_value_heads = n_head
    config.hidden_size         = n_embd
    config.head_dim            = n_embd // n_head
    config.intermediate_size   = n_embd * 4
    config.torch_dtype = "float32"
    config.vocab_size  = tok.vocab_size
    config.pad_token_id = tok.convert_tokens_to_ids("[PAD]")
    config.bos_token_id = tok.convert_tokens_to_ids("[BOS]")
    config.eos_token_id = tok.convert_tokens_to_ids("[EOS]")

    mod = __import__(f"modeling_rmt.huggingface_rmm_{module}", fromlist=["x"])
    RecurrentMemoryBase   = mod.RecurrentMemoryBase
    RecurrentMemoryConfig = mod.RecurrentMemoryConfig

    head_dim = state_size // n_head                      # the run scripts' convention
    rmm_config = RecurrentMemoryConfig(
        base_model_config=config, fla_layer_name=fla_layer_name,
        num_heads=n_head, head_dim=head_dim, expand_v=expand_v,
        conv_size=conv_kernel, use_short_conv=True,
        num_memory_vectors=num_memory_vectors, write_mode=write_mode, read_mode=read_mode,
        write_value_dim=None, num_memory_heads=num_memory_heads,
        use_parallel_prefill=True, max_n_segments=10,
        think_token_id=tok.convert_tokens_to_ids("[THINK]"),
        answer_token_id=tok.convert_tokens_to_ids("[ANSWER]"),
        bos_token_id=tok.convert_tokens_to_ids("[BOS]"),
        eos_token_id=tok.convert_tokens_to_ids("[EOS]"))
    model = RecurrentMemoryBase(rmm_config)
    return model, rmm_config, dict(requested_state_size=state_size, requested_num_heads=n_head,
                                   requested_head_dim=head_dim, requested_num_memory_heads=num_memory_heads)

In [18]:
def _layers(bm):
    return bm.model.layers if hasattr(bm,"model") else bm.transformer.h

def _first_wrapped_layer(model):
    for m in model.modules():
        if m.__class__.__name__ == "RecurrentMemoryCell":
            return list(_layers(m.model))[0]
    raise AssertionError("no RecurrentMemoryCell found")

def inspect_model(model):
    """Read ACTUAL heads/dims/state off live modules (robust across v5p7/v6p2/v6p3)."""
    L0 = _first_wrapped_layer(model)
    out = {}
    ba = getattr(L0, "base_layer", None)
    sa = getattr(ba, "self_attn", None) if ba is not None else None
    if sa is not None:
        out["transformer"] = dict(num_heads=sa.config.num_attention_heads,
                                  head_dim=getattr(sa,"head_dim",None),
                                  hidden=sa.config.hidden_size)
    g = L0.fla_layer
    gdn = {a:getattr(g,a) for a in ["num_heads","num_v_heads","head_dim","head_k_dim",
            "head_v_dim","key_dim","value_dim","hidden_size","expand_v","conv_size"] if hasattr(g,a)}
    nh, hk, hv = gdn.get("num_heads"), gdn.get("head_k_dim"), gdn.get("head_v_dim")
    if nh and hk and hv: gdn["recurrent_state_numel_per_layer"] = nh*hk*hv
    out["gdn"] = gdn
    c = getattr(L0, "compress", None)
    out["compress"] = None if c is None else dict(cls=c.__class__.__name__, mode=getattr(c,"mode",None),
        num_vectors=getattr(c,"num_vectors",None), write_value_dim=getattr(c,"write_value_dim",None),
        multi_head=hasattr(c,"num_heads"), num_heads=getattr(c,"num_heads",None),
        write_queries_shape=tuple(c.write_queries.shape) if hasattr(c,"write_queries") else None)
    d = getattr(L0, "decompress", None)
    out["decompress"] = None if d is None else dict(cls=d.__class__.__name__, mode=getattr(d,"mode",None),
        num_heads=getattr(getattr(d,"cross_attn",None),"num_heads",None))
    rq = getattr(L0, "read_queries", None)
    out["read_queries_shape"] = None if rq is None else tuple(rq.shape)
    return out

In [19]:
def check_knobs(rmm_config, model, requested):
    """SAFETY NET: replay the model\'s kwarg-filter and compare requested vs actual GDN."""
    L0 = _first_wrapped_layer(model)
    layer_cls = type(L0.fla_layer)
    sig = inspect.signature(layer_cls.__init__)
    produced = rmm_config.fla_layer_kwargs()
    accepted = {k:v for k,v in produced.items() if k in sig.parameters and k not in {"self","hidden_size","layer_idx"}}
    dropped  = {k:v for k,v in produced.items() if k not in accepted}
    print(f"layer_cls(GDN)        : {layer_cls.__name__}")
    print(f"layer_cls.__init__ sig: {list(sig.parameters)}")
    print(f"fla_layer_kwargs()    : {produced}")
    print(f"  ACCEPTED via THIS sig: {accepted}")
    print(f"  DROPPED via THIS sig : {dropped}")
    g = inspect_model(model)["gdn"]
    print("REQUESTED:", requested)
    print("ACTUAL gdn:", {k:g.get(k) for k in ["num_heads","head_dim","head_k_dim","head_v_dim","recurrent_state_numel_per_layer"]})
    mism = [k for k,a in [("num_heads",g.get("num_heads")),("head_dim",g.get("head_dim"))]
            if {"num_heads":requested["requested_num_heads"],"head_dim":requested["requested_head_dim"]}[k] != a]
    print(("\u274c MISMATCH on "+str(mism)+": GDN NOT using requested config (silent fallback).") if mism
          else "\u2705 GDN honors requested num_heads & head_dim.")
    return g

## ARBITER (point 4) — v6p2, ss16 vs ss32 (H4). If GDN is identical, `state_size` is dead.

In [20]:
for ss in [16, 32]:
    print("="*70, f"\n v6p2  state_size={ss}  n_head=4")
    model, cfg, req = build_model(state_size=ss, n_head=4, module="v6p2")
    check_knobs(cfg, model, req); del model

 v6p2  state_size=16  n_head=4
layer_cls(GDN)        : ReadAwareGatedDeltaNet
layer_cls.__init__ sig: ['self', 'args', 'kwargs']
fla_layer_kwargs()    : {'num_heads': 4, 'head_dim': 4, 'state_size': 32, 'expand': 2.0, 'conv_size': 4, 'conv_kernel': 4, 'use_short_conv': True}
  ACCEPTED via THIS sig: {}
  DROPPED via THIS sig : {'num_heads': 4, 'head_dim': 4, 'state_size': 32, 'expand': 2.0, 'conv_size': 4, 'conv_kernel': 4, 'use_short_conv': True}
REQUESTED: {'requested_state_size': 16, 'requested_num_heads': 4, 'requested_head_dim': 4, 'requested_num_memory_heads': 1}
ACTUAL gdn: {'num_heads': 6, 'head_dim': 256, 'head_k_dim': 256, 'head_v_dim': 512, 'recurrent_state_numel_per_layer': 786432}
❌ MISMATCH on ['num_heads', 'head_dim']: GDN NOT using requested config (silent fallback).
 v6p2  state_size=32  n_head=4
layer_cls(GDN)        : ReadAwareGatedDeltaNet
layer_cls.__init__ sig: ['self', 'args', 'kwargs']
fla_layer_kwargs()    : {'num_heads': 4, 'head_dim': 8, 'state_size': 32, 'ex

## CONTROL — v5p7 honors `head_dim` (the real `GatedDeltaNet` class, named signature)

v5p7 builds the GDN via `getattr(fla.layers, name)` (named `__init__`), so the
`inspect.signature` filter keeps `head_dim`/`num_heads`. Expect `head_k_dim` to
**change** between ss16 (→4) and ss32 (→8) — the difference you saw in old runs.

In [21]:
for ss in [16, 32]:
    print("="*70, f"\n v5p7  state_size={ss}  n_head=4  (write=pool, read=unpool)")
    try:
        model, cfg, req = build_model(state_size=ss, n_head=4, module="v5p7",
                                      write_mode="pool", read_mode="unpool")
        check_knobs(cfg, model, req); del model
    except Exception as e:
        print("v5p7 build error:", type(e).__name__, e)

 v5p7  state_size=16  n_head=4  (write=pool, read=unpool)
layer_cls(GDN)        : GatedDeltaNet
layer_cls.__init__ sig: ['self', 'hidden_size', 'expand_v', 'head_dim', 'num_heads', 'num_v_heads', 'mode', 'use_gate', 'use_short_conv', 'allow_neg_eigval', 'conv_size', 'conv_bias', 'layer_idx', 'norm_eps', 'kwargs']
fla_layer_kwargs()    : {'num_heads': 4, 'head_dim': 4, 'state_size': 32, 'expand': 2.0, 'conv_size': 4, 'conv_kernel': 4, 'use_short_conv': True}
  ACCEPTED via THIS sig: {'num_heads': 4, 'head_dim': 4, 'conv_size': 4, 'use_short_conv': True}
  DROPPED via THIS sig : {'state_size': 32, 'expand': 2.0, 'conv_kernel': 4}
REQUESTED: {'requested_state_size': 16, 'requested_num_heads': 4, 'requested_head_dim': 4, 'requested_num_memory_heads': 1}
ACTUAL gdn: {'num_heads': 4, 'head_dim': 4, 'head_k_dim': 4, 'head_v_dim': 8, 'recurrent_state_numel_per_layer': 128}
✅ GDN honors requested num_heads & head_dim.
 v5p7  state_size=32  n_head=4  (write=pool, read=unpool)
layer_cls(GDN)     

## FIX — v6p3 honors both knobs, hard-errors on silent drops, logs geometry

Expect `\u2705` for both ss, and `recurrent_state_numel_per_layer` to scale (128 vs 512).

In [22]:
for ss in [16, 32]:
    print("="*70, f"\n v6p3  state_size={ss}  n_head=4")
    model, cfg, req = build_model(state_size=ss, n_head=4, module="v6p3")
    check_knobs(cfg, model, req); del model

 v6p3  state_size=16  n_head=4
[RMM v6p3] GDN resolved: num_heads=4 head_k_dim=4 head_v_dim=8 state_numel/layer=128 (requested num_heads=4, head_dim=4)
layer_cls(GDN)        : ReadAwareGatedDeltaNet
layer_cls.__init__ sig: ['self', 'args', 'kwargs']
fla_layer_kwargs()    : {'num_heads': 4, 'head_dim': 4, 'expand_v': 2.0, 'conv_size': 4, 'use_short_conv': True}
  ACCEPTED via THIS sig: {}
  DROPPED via THIS sig : {'num_heads': 4, 'head_dim': 4, 'expand_v': 2.0, 'conv_size': 4, 'use_short_conv': True}
REQUESTED: {'requested_state_size': 16, 'requested_num_heads': 4, 'requested_head_dim': 4, 'requested_num_memory_heads': 1}
ACTUAL gdn: {'num_heads': 4, 'head_dim': 4, 'head_k_dim': 4, 'head_v_dim': 8, 'recurrent_state_numel_per_layer': 128}
✅ GDN honors requested num_heads & head_dim.
 v6p3  state_size=32  n_head=4
[RMM v6p3] GDN resolved: num_heads=4 head_k_dim=8 head_v_dim=16 state_numel/layer=512 (requested num_heads=4, head_dim=8)
layer_cls(GDN)        : ReadAwareGatedDeltaNet
layer_cl

## Side-by-side: GDN geometry across versions (same request)

In [23]:
def compare(state_size=32, n_head=4):
    for module, kw in [("v5p7", dict(write_mode="pool", read_mode="unpool")),
                       ("v6p2", {}), ("v6p3", {})]:
        try:
            m,_,_ = build_model(state_size=state_size, n_head=n_head, module=module, **kw)
            g = inspect_model(m)["gdn"]; del m
            print(f"{module}: num_heads={g.get('num_heads')} head_k_dim={g.get('head_k_dim')} "
                  f"head_v_dim={g.get('head_v_dim')} state/layer={g.get('recurrent_state_numel_per_layer')}")
        except Exception as e:
            print(f"{module}: ERROR {type(e).__name__}: {e}")

print("--- request ss16 H4 ---"); compare(16, 4)
print("--- request ss32 H4 ---"); compare(32, 4)

--- request ss16 H4 ---
v5p7: num_heads=4 head_k_dim=4 head_v_dim=8 state/layer=128
v6p2: num_heads=6 head_k_dim=256 head_v_dim=512 state/layer=786432
[RMM v6p3] GDN resolved: num_heads=4 head_k_dim=4 head_v_dim=8 state_numel/layer=128 (requested num_heads=4, head_dim=4)
v6p3: num_heads=4 head_k_dim=4 head_v_dim=8 state/layer=128
--- request ss32 H4 ---
v5p7: num_heads=4 head_k_dim=8 head_v_dim=16 state/layer=512
v6p2: num_heads=6 head_k_dim=256 head_v_dim=512 state/layer=786432
[RMM v6p3] GDN resolved: num_heads=4 head_k_dim=8 head_v_dim=16 state_numel/layer=512 (requested num_heads=4, head_dim=8)
v6p3: num_heads=4 head_k_dim=8 head_v_dim=16 state/layer=512
